> **Rama:** `feautre/feature-eng`
>
> Continúa a partir de `feature/eda` (usa el mismo `df` cargado y limpiado, y las conclusiones del EDA). Incluye el arranque de la Parte 2 (entendimiento del problema, carga de datos) y la tipificación/selección de variables.

# Parte 2 — Modelado

A partir de aquí se retoman las conclusiones de la Parte 1 (EDA) para construir el pipeline de
preprocesamiento, comparar modelos, ajustar hiperparámetros y evaluar el modelo final.

## **Paso 1: Entendimiento del problema**

**Problema de negocio:** las universidades quieren identificar, **en el momento de la matrícula**,
qué estudiantes tienen mayor riesgo de abandonar los estudios, para poder activar medidas de apoyo
tempranas (becas, tutorías, planes de pago flexibles, seguimiento académico).

**Tipo de problema:** clasificación multiclase (3 clases: `Dropout`, `Enrolled`, `Graduate`).

**Decisión clave — ventana temporal del modelo:** el EDA (Parte 1 de este mismo notebook) detectó, con ANOVA, que las 4 variables con
mayor F-estadístico de todo el dataset son las de `Curricular units 1st/2nd sem (approved/grade)`
(F hasta 1410), y lo confirmó con un ejemplo muy claro: la mediana de asignaturas aprobadas en 2º
semestre es **0 para Dropout** y **6 para Graduate**. Esto es **data leakage**: esas variables solo
se conocen *después* de que el estudiante ya ha cursado uno o dos semestres, es decir, cuando gran
parte de la ventana útil para intervenir ya se ha cerrado. El EDA añade además que estas 12 variables
están muy correlacionadas entre sí (r=0.94 en `credited` y `enrolled` 1º vs 2º semestre, r=0.90 en
`approved`, r≈0.68-0.77 dentro del mismo semestre), por lo que ni siquiera aportarían 12 señales
independientes si se usaran parcialmente.

Como el objetivo de negocio es actuar **lo antes posible**, el modelo final de este notebook se
construye **únicamente con variables disponibles en el momento de la matrícula** (datos
sociodemográficos, académicos previos y económicos), excluyendo por completo las 12 variables de
`Curricular units`. Más abajo (Paso 5) se cuantifica el coste de esta decisión en rendimiento, para
dejar constancia razonada del trade-off.

**Métrica de evaluación:** el target tiene un desbalance moderado (49.9% Graduate, 32.1% Dropout,
17.9% Enrolled, según el EDA). Usar `accuracy` premiaría a un modelo que ignorase la clase minoritaria
(`Enrolled`), así que la métrica principal de todo el proceso —selección de modelo, validación
cruzada y optimización— será **F1 macro**, que pondera igual a las tres clases.

In [3]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (classification_report, f1_score, confusion_matrix,
                              ConfusionMatrixDisplay)
from scipy.stats import loguniform

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

_art_eda = joblib.load('src/artifacts/01_eda.joblib')
df = _art_eda['df']

RANDOM_STATE = 42
sns.set_style('whitegrid')

## **Paso 2: Carga de datos**

El fichero viene separado por `;` y con decimales en punto. No hay nulos —el EDA confirma **0% de
nulos en todas las columnas**—, así que no aplicamos ninguna imputación.

El `DataFrame` `df` y la limpieza de nombres de columna (`.str.strip()`, necesaria porque el CSV
trae un tabulador pegado al final de `"Daytime/evening attendance\t"`) ya se hicieron en la
Parte 1 (EDA) de este mismo notebook, así que aquí simplemente lo reutilizamos.

In [4]:
# El DataFrame `df` ya se cargó y se limpió (columnas .str.strip()) en la Parte 1 (EDA).
# Lo reutilizamos aquí tal cual, sin volver a leer el CSV.
print(f'Filas: {df.shape[0]}  |  Columnas: {df.shape[1]}')
df.head()

Filas: 4424  |  Columnas: 37


,Marital status,Application mode,Application order,Course,Daytime/evening attendance,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


In [5]:
df['Target'].value_counts(normalize=True).round(3) * 100

Target
Graduate    49.9
Dropout     32.1
Enrolled    17.9
Name: proportion, dtype: float64

## **Paso 3: Tipificación manual de variables**

El EDA ya probó la función automática `tipifica_variables` (basada solo en umbrales de cardinalidad)
y comprobó que se equivoca en varios casos porque no conoce el significado real de cada variable:

- Marca como **"Numérica Discreta"** columnas que en realidad son **categóricas nominales**:
  `Application mode`, `Course`, `Previous qualification`, `Nacionality`,
  `Mother's/Father's qualification`, `Mother's/Father's occupation` (son códigos de categoría, no
  cantidades).
- Marca como **"Numérica Discreta"** variables que son en realidad **continuas**: `Admission grade`
  y las `Curricular units (grade)` de ambos semestres.
- Marca como **"Categórica"** variables que son **continuas** (`Unemployment rate`, `Inflation rate`,
  `GDP`, `Curricular units 2nd sem (without evaluations)`), simplemente porque tienen pocos valores
  únicos (9-10): son indicadores macroeconómicos por año/cohorte de matrícula, no por estudiante.

Por eso, igual que en el EDA, tipificamos a mano siguiendo el diccionario oficial del dataset en
lugar de confiar en la clasificación automática:

- **Numéricas continuas (18):** notas, edad, las 12 variables de `Curricular units` y los 3
  indicadores macroeconómicos (`Unemployment rate`, `Inflation rate`, `GDP`).
- **Categóricas (18):** códigos (estado civil, vía de acceso, curso, cualificaciones/ocupaciones de
  los padres) y las variables binarias 0/1 (`Debtor`, `Gender`, etc.).

In [6]:
numeric_all = [
    'Previous qualification (grade)', 'Admission grade', 'Age at enrollment',
    'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)',
    'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)',
    'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)',
    'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (enrolled)',
    'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (approved)',
    'Curricular units 2nd sem (grade)', 'Curricular units 2nd sem (without evaluations)',
    'Unemployment rate', 'Inflation rate', 'GDP'
]

categorical_all = [
    'Marital status', 'Application mode', 'Application order', 'Course',
    'Daytime/evening attendance', 'Previous qualification', 'Nacionality',
    "Mother's qualification", "Father's qualification", "Mother's occupation",
    "Father's occupation", 'Displaced', 'Educational special needs', 'Debtor',
    'Tuition fees up to date', 'Gender', 'Scholarship holder', 'International'
]

assert len(numeric_all) == 18 and len(categorical_all) == 18
assert set(numeric_all) | set(categorical_all) | {'Target'} == set(df.columns)
print('Tipificación manual OK: 18 numéricas + 18 categóricas + target (misma tipificación que en el EDA)')

Tipificación manual OK: 18 numéricas + 18 categóricas + target (misma tipificación que en el EDA)


## **Paso 4: Selección de variables (leakage y relevancia)**

Aplicamos las conclusiones del EDA, que ejecutó ANOVA para las 18 numéricas y Chi-cuadrado para las
18 categóricas, ambas contra el target:

1. **Excluimos las 12 variables `Curricular units 1st/2nd sem`**: son la causa del data leakage
   (Paso 1) y además están muy correlacionadas entre sí (r=0.94 en `credited`/`enrolled` 1º vs 2º
   semestre, r=0.90 en `approved`), por lo que ni siquiera aportarían información independiente si
   las usáramos parcialmente.
2. **Excluimos las variables no significativas según el EDA**:
   - `Inflation rate` (ANOVA, p=0.175 — la única numérica no significativa de las 18).
   - `Educational special needs` (Chi2, p=0.725) e `International` (Chi2, p=0.527), ambas con muy
     pocos casos positivos.
   - `Nacionality` (Chi2, p=0.242), con un test poco fiable: el 97.5% de los estudiantes (4314 de
     4424) son de una única nacionalidad y el resto se reparte en 20 nacionalidades con 1-2 casos
     cada una, lo que da frecuencias esperadas demasiado bajas para el test.
3. **Nos quedamos con el resto**, incluyendo las variables que el EDA señaló como más relevantes:
   - Numéricas "limpias": `Age at enrollment` (F=154.71, la más destacada del grupo sin leakage),
     `Admission grade` (F=35.65), `Previous qualification (grade)` (F=27.73).
   - Categóricas: `Tuition fees up to date` (Chi2=823.55, la variable categórica más relevante de
     todo el dataset), `Scholarship holder` (Chi2=409.94), `Debtor` (Chi2=259.33), además de `Course`
     y `Application mode`, también con relación fuerte con el target.
   - Macroeconómicas de cohorte: `Unemployment rate` y `GDP`, significativas pero con relación débil
     (F≈5-6 según el EDA); se mantienen porque siguen siendo información legítima disponible en el
     momento de matrícula (el año de ingreso), a diferencia de `Inflation rate`, que no llegó a ser
     significativa.

In [7]:
leakage_cols = [c for c in df.columns if 'Curricular units' in c]
non_significant_cols = ['Nacionality', 'Educational special needs', 'International', 'Inflation rate']
drop_cols = leakage_cols + non_significant_cols

X = df.drop(columns=drop_cols + ['Target'])
y = df['Target']

print(f'Variables descartadas ({len(drop_cols)}): {drop_cols}')
print(f'\nVariables finales del modelo ({X.shape[1]}): {list(X.columns)}')

Variables descartadas (16): ['Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)', 'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (enrolled)', 'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (approved)', 'Curricular units 2nd sem (grade)', 'Curricular units 2nd sem (without evaluations)', 'Nacionality', 'Educational special needs', 'International', 'Inflation rate']

Variables finales del modelo (20): ['Marital status', 'Application mode', 'Application order', 'Course', 'Daytime/evening attendance', 'Previous qualification', 'Previous qualification (grade)', "Mother's qualification", "Father's qualification", "Mother's occupation", "Father's occupation", 'Admission grade', 'Displaced', 'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder', 'Age at

**Coste de excluir el leakage (evidencia rápida).** Para dejar constancia razonada del
trade-off, comparamos el F1 macro (validación cruzada, modelo de referencia) *con* y *sin* las
variables de `Curricular units`. Esto NO forma parte del pipeline final, es solo la justificación
numérica de la decisión anterior.

In [8]:
_pre_check_leak = ColumnTransformer([
    ('num', StandardScaler(), [c for c in numeric_all if c not in non_significant_cols]),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
     [c for c in categorical_all if c not in non_significant_cols])
])
_pipe_with_leak = Pipeline([('pre', _pre_check_leak),
                             ('clf', LogisticRegression(max_iter=3000, class_weight='balanced'))])
_X_with_leak = df.drop(columns=non_significant_cols + ['Target'])
_skf_check = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
_score_with_leak = cross_validate(_pipe_with_leak, _X_with_leak, y, cv=_skf_check, scoring='f1_macro')['test_score'].mean()

_pipe_no_leak = Pipeline([
    ('pre', ColumnTransformer([
        ('num', StandardScaler(), [c for c in numeric_all if c in X.columns]),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False),
         [c for c in categorical_all if c in X.columns])
    ])),
    ('clf', LogisticRegression(max_iter=3000, class_weight='balanced'))
])
_score_no_leak = cross_validate(_pipe_no_leak, X, y, cv=_skf_check, scoring='f1_macro')['test_score'].mean()

print(f'F1 macro CON  Curricular units (leakage): {_score_with_leak:.3f}')
print(f'F1 macro SIN  Curricular units (leakage): {_score_no_leak:.3f}')
print(f'Diferencia: {_score_with_leak - _score_no_leak:.3f}')

F1 macro CON  Curricular units (leakage): 0.713
F1 macro SIN  Curricular units (leakage): 0.566
Diferencia: 0.147


Como se esperaba, incluir las variables de `Curricular units` infla artificialmente el
rendimiento (el modelo básicamente "ve" el resultado académico antes de predecirlo). Confirmamos la
decisión: el pipeline final se construye **sin** esas variables, priorizando un modelo honesto y útil
para intervención temprana sobre uno con una métrica más alta pero poco realista en producción.

In [9]:
import os
os.makedirs('src/artifacts', exist_ok=True)

joblib.dump({
    'X': X,
    'y': y,
    'numeric_all': numeric_all,
    'categorical_all': categorical_all
}, 'src/artifacts/02_feature_eng.joblib')
print("Feature Engineering guardado exitosamente.")

Feature Engineering guardado exitosamente.
